# Praetor interactive walkthrough

**Post-detection disposition engine.** An alert already fired; Praetor decides what happens next.

> The model recommends. The system authorizes.

Run the cells below, then use the dial to pick a mechanism. Every selection
destroys the previous throwaway SQLite store, activates a clean org config,
wires exactly one precondition, and runs only that scenario. The model provider
and ticket stamp are deterministic stand-ins; **everything downstream is the
real engine.**

Prefer a zero-setup version? The same ten scenarios are pre-rendered as a
clickable page under `demo/index.html`.


In [1]:
# Setup (hidden): make notebooks/ importable and load the shared scenario registry.
import sys
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "configs" / "example_org.yaml").exists():
            return candidate
    raise RuntimeError("run this notebook from inside the Praetor repo")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "notebooks"))

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

from walkthrough_scenarios import (
    SCENARIOS,
    close_scenario_store,
    run_scenario,
    scenario_session,
)

print("Praetor repo:", REPO)
print(f"scenarios loaded: {len(SCENARIOS)}")


Praetor repo: C:\Users\oalan\Praetor
scenarios loaded: 10


## Pick a mechanism


In [2]:
scenario_picker = widgets.RadioButtons(
    options=[(scenario.label, key) for key, scenario in SCENARIOS.items()],
    value=next(iter(SCENARIOS)),
    description="Scenario:",
    layout=widgets.Layout(width="max-content"),
    style={"description_width": "initial"},
)
scenario_output = widgets.Output()


def refresh_selected_scenario(change) -> None:
    if change.get("name") != "value" or change.get("new") is None:
        return
    with scenario_output:
        clear_output(wait=True)
        run_scenario(change["new"], show=lambda text: display(Markdown(text)))


scenario_picker.observe(refresh_selected_scenario, names="value")
display(widgets.VBox([scenario_picker, scenario_output]))
with scenario_output:
    run_scenario(scenario_picker.value, show=lambda text: display(Markdown(text)))
print("INTERACTIVE PICKER READY")


INTERACTIVE PICKER READY


## Scenario reference

Full setup wiring and engine code for every dial lives in
`notebooks/walkthrough_scenarios.py`. Each scenario asserts its own expected
disposition and fault flags, so a behavior change fails CI rather than quietly
rewriting the demo.


In [3]:
for key, scenario in SCENARIOS.items():
    display(Markdown(f"**`{key}`** — {scenario.headline}"))


**`earned_auto_contain`** — Containment is authorized, so a bounded directive is emitted.

**`benign_review`** — The safe floor: a human still sees it, nothing is isolated.

**`never_contain`** — A live never-contain entry overrides an explicit allow rule.

**`insufficient_corroboration`** — A single thin citation cannot earn containment.

**`not_allowlisted`** — Perfect evidence still loses to a missing allow rule.

**`rate_limit`** — Blast-radius ceiling refuses an otherwise valid containment.

**`circuit_breaker`** — A tripped containment breaker blocks all new isolation.

**`progressive_report`** — Read-only override rates, never an automatic policy promotion.

**`similar_case_exemplars`** — Past confirmed cases inform the prompt without granting power.

**`statute_curation`** — Proposed policy edits are refused at preflight by design.

In [4]:
print("CI SCENARIO SWEEP START")
with scenario_session():
    for scenario_name in SCENARIOS:
        run_scenario(scenario_name, show=lambda text: None)
    print("CI SCENARIO SWEEP COMPLETE")


CI SCENARIO SWEEP START


SCENARIO START: earned_auto_contain
alert             : ALERT-MAL-001
scenario beat     : earned auto_contain
model proposed    : auto_contain
PRAETOR DECIDED   : AUTO_CONTAIN
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : d7c73e0f35129d198d00d9a500f2b6341b263fe8d9afa3e3ac7e4168c3088f17
  >> CONTAINMENT DIRECTIVE EMITTED
     target          : host:WORKSTATION1 scope=host-isolation
     lifetime        : 300s  (hard cap 300)
     status          : emitted
     idempotency_key : 9a5b37471a29425a758091e05771eb7803f985a86e1dc8476f7fdcc10c62fe7c
     live_nc_hash    : 4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945
SCENARIO COMPLETE: earned_auto_contain


SCENARIO START: benign_review
alert             : ALERT-BEN-001
scenario beat     : safe human-review floor
model proposed    : standard_review
PRAETOR DECIDED   : STANDARD_REVIEW
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : 634c8e4c53db62ebcb8107bbeb358046105b480f6854f85ae5ce1500fbd1187f
  >> no containment directive - nothing isolated
SCENARIO COMPLETE: benign_review


SCENARIO START: never_contain
alert             : ALERT-DC-001
scenario beat     : live never-contain wins over allow
model proposed    : auto_contain
PRAETOR DECIDED   : ESCALATE
fault_flags       : ['never_contain_live_conflict']
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : d07759dd2d32a3abacc307ca639883e58dbaf6d3086d397092190192906338e4
  >> no containment directive - nothing isolated
SCENARIO COMPLETE: never_contain


SCENARIO START: insufficient_corroboration
alert             : ALERT-THIN-001
scenario beat     : corroboration floor
model proposed    : auto_contain
PRAETOR DECIDED   : ESCALATE
fault_flags       : ['insufficient_corroboration']
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : aa8a6284b136cae78f4919114241203139084b8a934223ec4c95fb7f6a5ab7fe
  >> no containment directive - nothing isolated
SCENARIO COMPLETE: insufficient_corroboration


...

SCENARIO START: rate_limit
alert             : ALERT-RATE-001
scenario beat     : transactional rate-limit refusal
model proposed    : auto_contain
PRAETOR DECIDED   : ESCALATE
fault_flags       : ['rate_limit_exceeded']
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : b4a8a927dda93caff3b67fef51b16ff356080e96f9028e7ea9162b67188f3a17
  >> no containment directive - nothing isolated
SCENARIO COMPLETE: rate_limit


SCENARIO START: circuit_breaker
alert             : ALERT-BREAKER-001
scenario beat     : containment circuit breaker
model proposed    : auto_contain
PRAETOR DECIDED   : ESCALATE
fault_flags       : ['containment_breaker_open']
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : dbdcbd7e6f16a6481111217d1678c59bda9db1195a780432ade2f317814d36be
  >> no containment directive - nothing isolated
SCENARIO COMPLETE: circuit_breaker


SCENARIO START: progressive_report
PROGRESSIVE AUTHORIZATION REPORT (read-only)
  read_only=True
  host/workstation: evals=3 overrides=1 override_rate=33%
SCENARIO COMPLETE: progressive_report


SCENARIO START: similar_case_exemplars
alert             : ALERT-EXEMPLAR-SOURCE
scenario beat     : seed a human-confirmed precedent
model proposed    : auto_contain
PRAETOR DECIDED   : AUTO_CONTAIN
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : 71ada79325da4be1ac4b9fc3e4bab905990494870c7f346958cef1325ae69f0c
  >> CONTAINMENT DIRECTIVE EMITTED
     target          : host:WORKSTATION-EXEMPLAR scope=host-isolation
     lifetime        : 300s  (hard cap 300)
     status          : emitted
     idempotency_key : 9c1e13f1dc0d96bc38908d1cbf11f5910c20098f9ceddc4797fc0e97cca62bcc
     live_nc_hash    : 4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945
prompt has prompt_exemplar_block: True
exemplar count: 1
  - precedent-1: auto_contain
    Confirmed office-to-PowerShell precedent. encoded powershell office parent scheduled IT automation A...
exemplar_scope: prompt_exemplar_block lists human-confirmed past cases for illustrati

SCENARIO START: statute_curation
alert             : ALERT-CURATION-SOURCE
scenario beat     : seed curation provenance
model proposed    : auto_contain
PRAETOR DECIDED   : AUTO_CONTAIN
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : 6b24d94a9aa4951fe5af4b989deefe8d02a2bf1af703242251c76a973163b8ad
  >> CONTAINMENT DIRECTIVE EMITTED
     target          : host:WORKSTATION-CURATION scope=host-isolation
     lifetime        : 300s  (hard cap 300)
     status          : emitted
     idempotency_key : a6915447c2b08c2d8d7fc060ca3e5e594d3a107a9c98fbde3f803675a3c0a6db
     live_nc_hash    : 4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945
artifact_kind     : proposed_statute
activation_status : proposed_for_review_only
preflight refused : proposed_artifact_not_activatable
pin: proposed_for_review_only
SCENARIO COMPLETE: statute_curation
CI SCENARIO SWEEP COMPLETE


In [5]:
close_scenario_store()
print("walkthrough scenario store closed")


walkthrough scenario store closed
